# SWAG Multiple Choice Finetune + Inference (MindSpore + MindHF)

**特性：**

- 流程：训练 -> 评估 -> 保存 -> 推理演示

本案例基于 MindSpore 2.7.1 和 MindHF 0.6.0 实现 BERT 模型在 SWAG 数据集上的多选任务微调。
## 1. 导入依赖与环境配置

导入必要的库，并设置 HuggingFace 镜像加速下载。


In [1]:
import os
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union

import numpy as np
import mindspore as ms

# ----------------------------
# Environment (HF-Mirror)
# ----------------------------
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


/root/.conda/envs/ms_py311/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/root/.conda/envs/ms_py311/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/root/.conda/envs/ms_py311/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/root/.conda/envs/ms_py311/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


'false'

## 2. 定义上下文设置函数

定义设置 MindSpore 运行环境（Ascend/PYNATIVE）和导入 MindHF 组件的辅助函数。


In [2]:
# ----------------------------
# MindSpore context
# ----------------------------
def set_ms_context():
    ms.set_seed(42)
    ms.set_context(mode=ms.PYNATIVE_MODE)
    try:
        ms.set_device("Ascend", 0)
    except AttributeError:
        ms.set_context(device_target="Ascend", device_id=0)


def to_numpy(x) -> np.ndarray:
    if isinstance(x, np.ndarray):
        return x
    if hasattr(x, "asnumpy"):
        return x.asnumpy()
    return np.asarray(x)


# ----------------------------
# Import mindhf.transformers
# ----------------------------
def import_mindhf_transformers():
    from mindhf.transformers import (
        AutoTokenizer,
        AutoModelForMultipleChoice,
        Trainer,
        TrainingArguments,
    )
    return AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments


## 3. 初始化与加载数据集

初始化运行环境，加载 SWAG 数据集，并根据配置截取训练集和验证集。


In [3]:
# 运行环境初始化
set_ms_context()
AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments = import_mindhf_transformers()

# 配置
model_checkpoint = "google-bert/bert-base-uncased"
output_dir = "./my_awesome_swag_model_ms"

# 样本量
max_train_samples = 2000
max_eval_samples = 1000

print(f">>> MindSpore: {ms.__version__}")
print(f">>> Device: Ascend:0 | Mode: PYNATIVE")
print(f">>> Model: {model_checkpoint}")

# 1. Dataset
from datasets import load_dataset
raw = load_dataset("swag", "regular")
raw["train"] = raw["train"].select(range(min(max_train_samples, len(raw["train"]))))
raw["validation"] = raw["validation"].select(range(min(max_eval_samples, len(raw["validation"]))))
print(f">>> Data: Train={len(raw['train'])}, Valid={len(raw['validation'])}")


/root/.conda/envs/ms_py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


>>> MindSpore: 2.7.1
>>> Device: Ascend:0 | Mode: PYNATIVE
>>> Model: google-bert/bert-base-uncased


>>> Data: Train=2000, Valid=1000


## 4. 数据预处理

加载 Tokenizer，将 SWAG 数据集的上下文与选项拼接，并进行编码（Tokenize）。


In [4]:
# 2. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

# 3. Preprocess
ending_names = ["ending0", "ending1", "ending2", "ending3"]

def preprocess_function(examples):
    first_sentences = [[c] * 4 for c in examples["sent1"]]
    question_headers = examples["sent2"]
    second_sentences = [
        [f"{h} {examples[end][i]}" for end in ending_names]
        for i, h in enumerate(question_headers)
    ]
    
    # Flatten
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(first_sentences, second_sentences, truncation=True)
    
    # Un-flatten
    result = {k: [v[i:i + 4] for i in range(0, len(v), 4)] for k, v in tokenized.items()}
    result["labels"] = examples["label"]
    return result

encoded = raw.map(preprocess_function, batched=True, remove_columns=raw["train"].column_names)



Map:   0%|                                                                                                                 | 0/2000 [00:00<?, ? examples/s]


Map:  50%|██████████████████████████████████████████████████                                                  | 1000/2000 [00:00<00:00, 1107.58 examples/s]


Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:01<00:00, 1376.03 examples/s]


Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:01<00:00, 1313.51 examples/s]


Map:   0%|                                                                                                                 | 0/1000 [00:00<?, ? examples/s]

...


Map:  85%|███████████████████████████████████████████████████████████████████████████████████▎              | 17000/20005 [00:11<00:02, 1428.53 examples/s]


Map:  90%|████████████████████████████████████████████████████████████████████████████████████████▏         | 18000/20005 [00:12<00:01, 1487.75 examples/s]


Map:  95%|█████████████████████████████████████████████████████████████████████████████████████████████     | 19000/20005 [00:13<00:00, 1521.57 examples/s]


Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████▉| 20000/20005 [00:13<00:00, 1402.11 examples/s]


Map: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 20005/20005 [00:13<00:00, 1435.17 examples/s]

## 5. 数据整理 (DataCollator)

定义数据整理器，负责动态 Padding 以及将数据转换为 MindSpore Tensor。


In [5]:
# ----------------------------
# DataCollator
# ----------------------------
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: Any
    padding: Union[bool, str] = "longest"
    pad_to_multiple_of: Optional[int] = None
    label_dtype: ms.dtype = ms.int32

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        label_key = "labels" if "labels" in features[0] else ("label" if "label" in features[0] else None)
        labels = [f.pop(label_key) for f in features] if label_key else None

        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flattened = []
        for feat in features:
            for i in range(num_choices):
                flattened.append({k: v[i] for k, v in feat.items()})

        # Try returning MindSpore tensors directly
        try:
            batch = self.tokenizer.pad(
                flattened,
                padding=self.padding,
                pad_to_multiple_of=self.pad_to_multiple_of,
                return_tensors="ms",
            )
            out = {k: v.reshape((batch_size, num_choices, -1)) for k, v in batch.items()}
            if labels is not None:
                out["labels"] = ms.Tensor(np.asarray(labels, dtype=np.int32), dtype=self.label_dtype)
            return out
        except Exception:
            # Fallback to numpy then convert
            batch_np = self.tokenizer.pad(
                flattened,
                padding=self.padding,
                pad_to_multiple_of=self.pad_to_multiple_of,
                return_tensors="np",
            )
            out = {}
            for k, v in batch_np.items():
                arr = np.asarray(v).reshape((batch_size, num_choices, -1))
                if k in ("input_ids", "attention_mask", "token_type_ids"):
                    arr = arr.astype(np.int32, copy=False)
                out[k] = ms.Tensor(arr)
            if labels is not None:
                out["labels"] = ms.Tensor(np.asarray(labels, dtype=np.int32), dtype=self.label_dtype)
            return out


## 6. 模型构建与训练配置

加载 `BertForMultipleChoice` 模型，配置 `TrainingArguments`，并初始化 `Trainer`。


In [6]:
# 4. Model
model = AutoModelForMultipleChoice.from_pretrained(model_checkpoint)

# 5. Metrics
def compute_metrics(eval_predictions):
    if hasattr(eval_predictions, "predictions"):
        logits = eval_predictions.predictions
        labels = eval_predictions.label_ids
    else:
        logits, labels = eval_predictions
    preds = np.argmax(to_numpy(logits), axis=1)
    labels = to_numpy(labels)
    return {"accuracy": float((preds == labels).mean())}

# 6. TrainingArguments (Fixed for current environment)

train_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",  
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=50,
    push_to_hub=False,
    remove_unused_columns=False,
    report_to=[],
)

# 7. Trainer
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForMultipleChoice(tokenizer),
    compute_metrics=compute_metrics,
)


[MS_ALLOC_CONF] config:  enable_vmm:True  vmm_align_size:2MB


Some weights of BertForMultipleChoice were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


/tmp/ipykernel_37813/2626949888.py:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


## 7. 执行训练与保存

启动训练流程，完成后进行评估并保存模型权重。


In [7]:
# 8. Run
print("\n>>> Starting training...")
trainer.train()

print("\n>>> Starting evaluation...")
metrics = trainer.evaluate()
print(f">>> Eval metrics: {metrics}")

# 9. Save
os.makedirs(output_dir, exist_ok=True)
trainer.save_model(output_dir)
try:
    tokenizer.save_pretrained(output_dir)
except Exception:
    pass
print(f">>> Model saved to: {output_dir}")


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



>>> Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.960300,0.883173,0.670000
2,0.372600,0.957089,0.679000
3,0.097600,1.163250,0.681000



>>> Starting evaluation...


>>> Eval metrics: {'eval_loss': 1.1632496118545532, 'eval_accuracy': 0.681, 'eval_runtime': 2.83, 'eval_samples_per_second': 353.355, 'eval_steps_per_second': 44.169, 'epoch': 3.0}


>>> Model saved to: ./my_awesome_swag_model_ms


## 8. 推理演示

从验证集中抽取样本，进行端到端的推理预测演示。


In [8]:
# 10. Inference Demo
print("\n>>> Running Inference Demo...")
sample = raw["validation"][0]
context = sample["sent1"]
header = sample["sent2"]
choices = [sample[e] for e in ending_names]

first = [context] * 4
second = [f"{header} {c}" for c in choices]

tok = tokenizer(first, second, truncation=True, padding=True, return_tensors="ms")
inputs = {k: v.reshape((1, 4, -1)) for k, v in tok.items()}

outputs = model(**inputs)
logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits
pred = int(np.argmax(logits.asnumpy(), axis=1)[0])

print("-" * 50)
print(f"Context: {context}")
print(f"Header : {header}")
for i, c in enumerate(choices):
    mark = "[x]" if i == pred else "[ ]"
    print(f"  {mark} {c}")
print("-" * 50)

gold = int(sample["label"])
if pred == gold:
    print(f"Result: CORRECT (Pred: {pred}, Gold: {gold})")
else:
    print(f"Result: INCORRECT (Pred: {pred}, Gold: {gold})")



>>> Running Inference Demo...
--------------------------------------------------
Context: Students lower their eyes nervously.
Header : She
  [ ] pats her shoulder, then saunters toward someone.
  [ ] turns with two students.
  [x] walks slowly towards someone.
  [ ] wheels around as her dog thunders out.
--------------------------------------------------
Result: CORRECT (Pred: 2, Gold: 2)
